In [ ]:
import os
from sklearn.pipeline import Pipeline
from model_class.feature_builder_clean import FeatureBuilder
from model_class.feature_builder_transformer import FeatureBuilderTransformer
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import IsolationForest
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import roc_auc_score, roc_curve
from feature_testing import main as feature_testing_main
from sklearn.model_selection import train_test_split
import time

### 1. Import data

In [ ]:
# Get the current directory
current_directory = os.getcwd()
    
# Get the source file path
file_path = f'{current_directory}/datasets/raw'
file = f"{file_path}/labelled_training_data.csv"
df_train = pd.read_csv(file, header=0)
y_train = df_train['evil']
X_train = df_train.drop(columns=['sus', 'evil'])

# Get the source file path
file_path = f'{current_directory}/datasets/raw'
file = f"{file_path}/labelled_testing_data.csv"
df_test = pd.read_csv(file, header=0)
y_test = df_test['evil']
X_test = df_test.drop(columns=['sus', 'evil'])

X_test_dev, X_test_final, y_test_dev, y_test_final = train_test_split(
    X_test,
    y_test,
    test_size=0.30,
    stratify=y_test,
    random_state=42
)

# Get the source file path
file_path = f'{current_directory}/datasets/raw'
file = f"{file_path}/labelled_validation_data.csv"
df_validation = pd.read_csv(file, header=0)
y_validation = df_validation['evil']
X_validation = df_validation.drop(columns=['sus', 'evil'])

In [ ]:
def evaluate_model(y_true, y_pred, dataset_type="Dataset", anomaly_scores=None):
    # Returns -1 for outliers and 1 for inliers.
    # change -1 to 0:
    y_pred = np.where(y_pred == -1, 1, 0)

    # Get unique values and their counts in y_true
    unique_values, counts = np.unique(y_true, return_counts=True)
    print(f"y_true_{dataset_type} Unique values: {unique_values}")
    print(f"y_true_{dataset_type} Counts of each value: {counts}")

    # Get unique values and their counts in y_pred
    unique_values, counts = np.unique(y_pred, return_counts=True)
    print(f"y_pred_{dataset_type} Unique values: {unique_values}")
    print(f"y_pred_{dataset_type} Counts of each value: {counts}")

     # Confusion Matrix
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Inlier", "Outlier"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f'Confusion Matrix for {dataset_type}')
    plt.show()

    if dataset_type == "Test Dataset":
        # Use continuous anomaly scores for a proper ROC curve.
        # Binary y_pred only has one threshold → single point → AUC ~0.50.
        if anomaly_scores is None:
            raise ValueError("Pass anomaly_scores=-model.score_samples(X_test) for a meaningful ROC curve.")
        roc_auc = roc_auc_score(y_true, anomaly_scores)
        fpr, tpr, _ = roc_curve(y_true, anomaly_scores)

        print(f"ROC AUC: {roc_auc:.4f}")
        plt.figure()
        plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title(f'Receiver Operating Characteristic for {dataset_type}')
        plt.legend(loc="lower right")
        plt.show()

### 2. build features (and find best ones using a testing split)

In [ ]:
scale_cols = [
    "stackAddresses_len",
    "stackAddresses_jump_std", 
    "stackAddresses_unique_ratio",
    "processId_eventId_past_freq",
    "processId_past_freq",
    "eventId_past_freq",
    "threadId_past_freq",
    "parentProcessId_past_freq",
    "userId_past_freq",
    "child_process_spawn_rate_so_far",
]

passthrough_cols = [
    "argsNum",
    "userId_binary",
    "returnValue",
    "returnValue_is_error",
    "args_has_path",
    "mountNamespace_binary",
    "same_process_name_as_parent",
    "same_user_as_parent",
    "is_parent_system_process",
    "is_system_process"
]

features_dir = os.path.join(current_directory, 'features_data')
existing_csvs = [f for f in os.listdir(features_dir) if f.endswith('.csv')] if os.path.exists(features_dir) else []

if existing_csvs:
    print(f"Found existing feature CSVs {existing_csvs}, skipping feature_testing_main()")
else:
    print("No feature CSVs found, running feature_testing_main()...")
    feature_testing_main()
    attempts = 0
    while not os.path.exists(os.path.join(current_directory, './features_data/best_features_dev_k10.csv')):
        print("Waiting for best_features_dev_k10.csv to be created...")
        time.sleep(1)  # Wait for 1 second before checking again
        attempts += 1
        if attempts >= 10:  # Wait for a maximum of 10 seconds
            print("Timeout: best_features_dev_k10.csv was not created within 10 seconds.")
            break
    


# Load best features (prefer top-10, fallback to top-5)
best_path = os.path.join(os.getcwd(), './features_data/best_features_dev_k10.csv')
if not os.path.exists(best_path):
    best_path = os.path.join(os.getcwd(), './features_data/best_features_dev_k5.csv')
if os.path.exists(best_path):
    top = pd.read_csv(best_path)['feature'].tolist()
    print('Loaded top features from', best_path)
    # filter scale and passthrough lists to only keep selected features
    scale_cols = [c for c in scale_cols if c in top]
    passthrough_cols = [c for c in passthrough_cols if c in top]
    print('Filtered scale_cols:', scale_cols)
    print('Filtered passthrough_cols:', passthrough_cols)
else:
    print('No best_features_dev_k10/5 file found; using original feature lists')
# Rebuild scaler with filtered columns
scaler = ColumnTransformer(
    transformers=[
        ('scale', RobustScaler(), scale_cols),
        ('pass', 'passthrough', passthrough_cols),
    ]
)

1. Isolation Forest

In [ ]:
model = Pipeline([
    ("features", FeatureBuilderTransformer(FeatureBuilder(), return_numpy=False)),
    ("scaler", scaler),
    ("iforest", IsolationForest(
            n_estimators=100,
            contamination="auto",
            max_features=0.7,
            random_state=2000
        ))

])

In [ ]:
# Perform fit on X _train and returns labels for y_pred.
print("Fitting model using training data...")
y_pred_train = model.fit_predict(X_train)

In [ ]:
# model.fit(X_train)
# scores = model.decision_function(X_validation)
# plt.figure(figsize=(10, 6))
# plt.hist(scores[y_validation == 0], bins=50, alpha=0.5, label='Inliers (y=0)')
# plt.hist(scores[y_validation == 1], bins=50, alpha=0.5, label='Outliers (y=1)')
# plt.axvline(x=threshold, color='red', linestyle='--', label=f'Threshold = {threshold}')
# plt.xlabel('Anomaly Score')
# plt.ylabel('Frequency')
# plt.title('Distribution of Anomaly Scores on Validation Set')
# plt.legend()
# plt.show()

In [ ]:
# Perform fit on X _train and returns labels for y_pred.
print("Predict on validation data...")
y_pred_validation= model.predict(X_validation)

# Perform fit on X _train and returns labels for y_pred.
print("Predict on final test split...")
y_pred_test = model.predict(X_test_final)

### Model Evaluation

In [ ]:
test_anomaly_scores = -model.score_samples(X_test_final)  # higher = more anomalous


evaluate_model(y_train, y_pred_train, dataset_type="Training Dataset")
evaluate_model(y_validation, y_pred_validation, dataset_type="Validation Dataset")
evaluate_model(y_test_final, y_pred_test, dataset_type="Test Dataset", anomaly_scores=test_anomaly_scores)

In [ ]:
anomaly_scores = model.score_samples(X_test_final)

anomaly_scores = -anomaly_scores  # Invert scores so that higher values indicate more anomalous instances
plt.hist(anomaly_scores, bins=100)
plt.title("Isolation Forest Anomaly Score Distribution")
plt.show()